# Segunda sesión de prácticas: árboles de Gentzen

**Paquetes para lógica en python**


| Paquete | Verifica teoremas | Devuelve pasos de la prueba | Estilo de prueba | Comentarios |
|----------|:----------------:|:--------------------------:|------------------|-------------|
| **PyProver** | ✅ | ❌ | Resolución automática | Muy sencillo para comprobar si una conclusión se sigue de unas premisas (`True` / `False`). No muestra derivaciones. |
| **nd-prover** | ✅ | ✅ | Deducción natural (Fitch) | La opción más cercana a una demostración paso a paso legible por humanos. |
| **Mathesis** | ✅ | ✅ | Deducción natural, tableaux y secuentes | API moderna y flexible para construir, manipular y visualizar pruebas. |
| **FLiP** | ✅ | ✅ | Deducción natural | Orientado a enseñanza y razonamiento formal con justificaciones explícitas. |
| **Logic Through Python** | ✅ | ✅ | Pruebas formales académicas | Excelente para estudiar lógica y trabajar con objetos de prueba detallados. |

 **Recomendación rápida**

| Necesidad | Paquete recomendado |
|------------|---------------------|
| Comprobar si una conclusión es válida | **PyProver** |
| Obtener una prueba paso a paso | **nd-prover** |
| Integrar lógica formal en una aplicación Python | **Mathesis** |
| Aprender o enseñar deducción natural | **FLiP** |
| Estudiar fundamentos de lógica y demostraciones | **Logic Through Python** |



## 🌲 Teoría de Demostración con Mathesis

<div class="alert alert-info">
<b>¿Qué es un Árbol de Demostración?</b><br>
Es una estructura gráfica que representa una inferencia lógica estructurada paso a paso. Fue introducido por el matemático <b>Gerhard Gentzen</b> para descomponer fórmulas complejas en partes más simples. Su objetivo es demostrar de forma mecánica si un argumento es totalmente válido.
</div>

---

### 🔑 Conceptos Clave

*   **Secuente**: Unidad básica de trabajo escrita formalmente como $\Gamma \vdash \Delta$.
*   **Torniquete ($\vdash$)**: Símbolo central de separación del secuente.
*   **Antecedente ($\Gamma$)**: Lado izquierdo del torniquete. Representa las premisas asumidas como verdaderas.
*   **Consecuente ($\Delta$)**: Lado derecho del torniquete. Representa la conclusión que se quiere demostrar.
*   **Estructura**: La demostración se construye apilando reglas de inferencia de forma jerárquica.
*   **Dirección**: La conclusión final se coloca en la raíz y se descompone hacia arriba abriendo ramas.
*   **Objetivo**: El argumento es válido cuando todas las ramas superiores terminan en un axioma inicial trivial (ej. $A \vdash A$).

---

### ⚙️ Mecánica de Construcción

Cada conectiva lógica ($\land$, $\lor$, $\rightarrow$, $\neg$) posee reglas específicas de descomposición según su posición:

*   **Reglas Izquierdas**: Descomponen los conectores ubicados en las premisas ($\Gamma$).
*   **Reglas Derechas**: Descomponen los conectores ubicados en la conclusión ($\Delta$).
*   **Bifurcación**: Reglas como la *Conjunción a la Derecha* ($\land R$) dividen el árbol en dos ramas independientes al exigir que se demuestren ambos componentes por separado.


### 📑 Guía de Reglas del Cálculo de Secuentes de Gentzen (`mathesis`)

| Conector / Tipo | Regla en Python | Acción Estructural | Ejemplo de Transformación |
| :--- | :--- | :--- | :--- |
| **($\rightarrow$)** | `rules.Conditional.Left()` | Rompe la implicación en las premisas. Abre **dos ramas**. | `P -> Q ⊢ R` <br> ── Extiende a ── <br> `⊢ P, R` **y** `Q ⊢ R` |
| | `rules.Conditional.Right()` | Mueve el antecedente del condicional a las premisas. | `⊢ P -> Q` $\implies$ `P ⊢ Q` |
| **($\land$)** | `rules.Conjunction.Left()` | Separa los componentes de una conjunción en premisas individuales. | `P ∧ Q ⊢ R` $\implies$ `P, Q ⊢ R` |
| | `rules.Conjunction.Right()` | Para demostrar ambas premisas juntas, abre **dos ramas** individuales. | `⊢ P ∧ Q` <br> ── Extiende a ── <br> `⊢ P` **y** `⊢ Q` |
| **($\lor$)** | `rules.Disjunction.Left()` | Análisis por casos. Evalúa la conclusión bajo ambas opciones en **dos ramas**. | `P ∨ Q ⊢ R` <br> ── Extiende a ── <br> `P ⊢ R` **y** `Q ⊢ R` |
| | `rules.Disjunction.Right()` | Aplana las opciones de la disyunción manteniéndolas en la conclusión. | `⊢ P ∨ Q` $\implies$ `⊢ P, Q` |
| **($\neg$)** | `rules.Negation.Left()` | Quita la negación izquierda y pasa la variable positiva a la derecha. | `¬P ⊢ Q` $\implies$ `⊢ P, Q` |
| | `rules.Negation.Right()` | Quita la negación derecha y pasa la variable positiva a la izquierda. | `P ⊢ ¬Q` $\implies$ `P, Q ⊢ ` |
| **Estructurales** | `rules.Structural.ContractionLeft()` | Fusiona premisas idénticas o duplicadas en una sola. | `P, P ⊢ Q` $\implies$ `P ⊢ Q` |
| | `rules.Structural.PermutationLeft()` | Cambia el orden físico de las premisas en la lista de la izquierda. | `P, Q ⊢ R` $\implies$ `Q, P ⊢ R` |
| | `rules.Structural.WeakeningRight()` | Introduce una variable irrelevante o arbitraria en las conclusiones. | `P ⊢ Q` $\implies$ `P ⊢ Q, R` |


In [1]:
from IPython.display import display, Math

## **Un ejemplo de juguete**
    P∧(P→Q)⇒Q

In [2]:
from mathesis.deduction.sequent_calculus.sequent_tree import SequentTree
from mathesis.deduction.sequent_calculus import rules
from mathesis.grammars import BasicGrammar
grammar = BasicGrammar()

In [3]:
premises = grammar.parse(["P ∧ (P → Q)"])
conclusions = grammar.parse(["Q"])

In [4]:
st = SequentTree(premises, conclusions)
Math(st[1].sequent.latex())

<IPython.core.display.Math object>

In [5]:
print("=== Árbol Inicial de Gentzen ===")
print(st.tree())

=== Árbol Inicial de Gentzen ===
P ∧ (P → Q) 1 ⇒ Q 2



In [6]:
st.apply(st[1], rules.Conjunction.Left())
print(st.tree())

P ∧ (P → Q) 1 ⇒ Q 2 [∧L]
└── P 3, P → Q 4 ⇒ Q 5



In [7]:
st.apply(st[4], rules.Conditional.Left())
print(st.tree())

P ∧ (P → Q) 1 ⇒ Q 2 [∧L]
└── P 3, P → Q 4 ⇒ Q 5 [→L]
    ├── P 7 ⇒ P 6, Q 8
    └── Q 9, P 10 ⇒ Q 11



## **Ley DeMorgan**
    ¬(P ∨ Q)  ⇒ ¬P ∧ ¬Q

In [8]:
premisa = grammar.parse(["¬(P ∨ Q)"])
conclusion = grammar.parse(["¬P ∧ ¬Q"])

In [9]:
# Inicializamos el árbol de Gentzen
st = SequentTree(premisa, conclusion)
print("=== 0. Estado Inicial del Secuente ===")
print(st.tree())

=== 0. Estado Inicial del Secuente ===
¬(P ∨ Q) 1 ⇒ ¬P ∧ ¬Q 2



In [10]:
# PASO 1: Rompemos la Conjunción (∧) del lado derecho (Right)
# Esto bifurcará el árbol para demostrar ¬P y ¬Q de manera aislada
st.apply(st[2], rules.Conjunction.Right())
print(st.tree())

¬(P ∨ Q) 1 ⇒ ¬P ∧ ¬Q 2 [∧R]
├── ¬(P ∨ Q) 4 ⇒ ¬P 3
└── ¬(P ∨ Q) 6 ⇒ ¬Q 5



In [11]:
# PASO 3: Pasamos la negación principal ¬(P ∨ Q) de la izquierda a la derecha
st.apply(st[4], rules.Negation.Left())
print(st.tree())

¬(P ∨ Q) 1 ⇒ ¬P ∧ ¬Q 2 [∧R]
├── ¬(P ∨ Q) 4 ⇒ ¬P 3 [¬L]
│   └──  ⇒ P ∨ Q 7, ¬P 8
└── ¬(P ∨ Q) 6 ⇒ ¬Q 5



In [12]:
st.apply(st[6], rules.Negation.Left())
print(st.tree())

¬(P ∨ Q) 1 ⇒ ¬P ∧ ¬Q 2 [∧R]
├── ¬(P ∨ Q) 4 ⇒ ¬P 3 [¬L]
│   └──  ⇒ P ∨ Q 7, ¬P 8
└── ¬(P ∨ Q) 6 ⇒ ¬Q 5 [¬L]
    └──  ⇒ P ∨ Q 9, ¬Q 10



In [13]:
# PASO 4: Expandimos la Disyunción (∨) a la derecha para aplanar P ∨ Q
st.apply(st[7], rules.Disjunction.Right())
print(st.tree())

¬(P ∨ Q) 1 ⇒ ¬P ∧ ¬Q 2 [∧R]
├── ¬(P ∨ Q) 4 ⇒ ¬P 3 [¬L]
│   └──  ⇒ P ∨ Q 7, ¬P 8 [∨R]
│       └──  ⇒ P 11, Q 12, ¬P 13
└── ¬(P ∨ Q) 6 ⇒ ¬Q 5 [¬L]
    └──  ⇒ P ∨ Q 9, ¬Q 10



In [14]:
st.apply(st[9], rules.Disjunction.Right())
print(st.tree())

¬(P ∨ Q) 1 ⇒ ¬P ∧ ¬Q 2 [∧R]
├── ¬(P ∨ Q) 4 ⇒ ¬P 3 [¬L]
│   └──  ⇒ P ∨ Q 7, ¬P 8 [∨R]
│       └──  ⇒ P 11, Q 12, ¬P 13
└── ¬(P ∨ Q) 6 ⇒ ¬Q 5 [¬L]
    └──  ⇒ P ∨ Q 9, ¬Q 10 [∨R]
        └──  ⇒ P 14, Q 15, ¬Q 16



In [15]:
# En el paso anterior ya hemos concluido, pero si lo queremos ver más claro
st.apply(st[13], rules.Negation.Right())
print(st.tree())

¬(P ∨ Q) 1 ⇒ ¬P ∧ ¬Q 2 [∧R]
├── ¬(P ∨ Q) 4 ⇒ ¬P 3 [¬L]
│   └──  ⇒ P ∨ Q 7, ¬P 8 [∨R]
│       └──  ⇒ P 11, Q 12, ¬P 13 [¬R]
│           └── P 17 ⇒ P 18, Q 19
└── ¬(P ∨ Q) 6 ⇒ ¬Q 5 [¬L]
    └──  ⇒ P ∨ Q 9, ¬Q 10 [∨R]
        └──  ⇒ P 14, Q 15, ¬Q 16



In [16]:
st.apply(st[16], rules.Negation.Right())
print(st.tree())

¬(P ∨ Q) 1 ⇒ ¬P ∧ ¬Q 2 [∧R]
├── ¬(P ∨ Q) 4 ⇒ ¬P 3 [¬L]
│   └──  ⇒ P ∨ Q 7, ¬P 8 [∨R]
│       └──  ⇒ P 11, Q 12, ¬P 13 [¬R]
│           └── P 17 ⇒ P 18, Q 19
└── ¬(P ∨ Q) 6 ⇒ ¬Q 5 [¬L]
    └──  ⇒ P ∨ Q 9, ¬Q 10 [∨R]
        └──  ⇒ P 14, Q 15, ¬Q 16 [¬R]
            └── Q 20 ⇒ P 21, Q 22

